In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
from torch import nn, optim

# CNN โมเดลพื้นฐาน
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
    def forward(self, x):
        return self.fc(self.conv(x))

# ฟังก์ชันสร้าง multi-augmentation ตามค่า magnitude
def build_transform(mag):
    return transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(
            brightness=mag,
            contrast=mag,
            saturation=mag
        ),
        transforms.RandomRotation(degrees=mag * 45),  # ปรับตาม mag
        transforms.RandomAffine(degrees=0, translate=(mag * 0.2, mag * 0.2)),
        transforms.ToTensor(),
        transforms.RandomErasing(p=mag * 0.5)
    ])




In [6]:
# Parameter Set Up
model = SimpleCNN()

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

total_epochs = 30

# ค่าเริ่มต้นของความแรง
magnitude = 0.1
last_val_loss = None

# Training Model
for epoch in range(total_epochs):

  # สร้าง transform ใหม่ตาม magnitude
  transform = build_transform(magnitude)

  # Dataset with Adaptive Transform
  trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
  trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

  model.train()
  total_loss = 0

  for images, labels in trainloader:
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()


  # แสดงค่า learning rate และ loss
  current_lr = optimizer.param_groups[0]['lr']
  print(f"Epoch {epoch+1}: LR = {current_lr:.6f}, Loss = {total_loss:.4f}")


  # Dataset for Validation Only
  valset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())
  valloader = DataLoader(valset, batch_size=64, shuffle=False)
  val_loss = 0

  # ประเมินบน validation set
  model.eval()
  val_loss = 0
  with torch.no_grad():
    for images, labels in valloader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      loss = criterion(outputs, labels)
      val_loss += loss.item()
    val_loss /= len(valloader)

  # ปรับ magnitude ตาม trend ของ validation loss
  if last_val_loss is not None:
    if val_loss > last_val_loss:  # โมเดลแย่ลง → ลดความแรง
      magnitude = max(0.05, magnitude - 0.05)
    else:  # โมเดลดีขึ้น → เพิ่มความแรง
      magnitude = min(0.6, magnitude + 0.05)
  last_val_loss = val_loss


Epoch 1: LR = 0.001000, Loss = 1285.9273
Epoch 2: LR = 0.001000, Loss = 1055.9353
Epoch 3: LR = 0.001000, Loss = 978.0699
Epoch 4: LR = 0.001000, Loss = 929.2464
Epoch 5: LR = 0.001000, Loss = 916.8001
Epoch 6: LR = 0.001000, Loss = 919.0911
Epoch 7: LR = 0.001000, Loss = 854.6223
Epoch 8: LR = 0.001000, Loss = 869.8755
Epoch 9: LR = 0.001000, Loss = 886.1702
Epoch 10: LR = 0.001000, Loss = 837.8419
Epoch 11: LR = 0.001000, Loss = 859.6033
Epoch 12: LR = 0.001000, Loss = 810.0184
Epoch 13: LR = 0.001000, Loss = 829.2921
Epoch 14: LR = 0.001000, Loss = 784.4774
Epoch 15: LR = 0.001000, Loss = 813.1688
Epoch 16: LR = 0.001000, Loss = 842.5635
Epoch 17: LR = 0.001000, Loss = 794.0153
Epoch 18: LR = 0.001000, Loss = 827.0069


KeyboardInterrupt: 